<a href="https://colab.research.google.com/github/dishantravi5936-ship-it/NER/blob/main/MusePose_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# Initial setup: Clone repository, install system deps, and basic Python packages

%cd /content
!git clone -b dev https://github.com/camenduru/MusePose
%cd /content/MusePose

# 1. Install system-level build tools and libraries for packages like xtcocotools and opencv-python-headless
# libgl1-mesa-glx, libsm6, libxext6 are often needed for graphical libraries like OpenCV
# python3-dev provides headers needed for compiling Python extensions like xtcocotools
!apt-get update -qq && apt-get install -y -qq build-essential libgl1-mesa-glx libsm6 libxext6 libxrender-dev python3-dev

# 2. Install Python build dependencies and headless OpenCV
# Uninstall existing opencv-python if present to prevent conflicts with headless version
!pip uninstall -y opencv-python opencv-contrib-python || true
!pip install -q opencv-python-headless # Headless version is better for Colab environments
!pip install -q cython # Needed for compiling some dependencies like xtcocotools
!pip install -q xtcocotools # Required for mmpose, often has build issues resolved by system deps + cython

# 3. Install other Python packages. Let Colab use its default PyTorch/CUDA setup.
# Removed explicit torch/xformers installation as they were causing conflicts.
# Removed diffusers, transformers, accelerate, einops, omegaconf from here to consolidate in next cell.
!pip install -q moviepy av ninja

# 4. Download pretrained weights and assets (kept as is)
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/MusePose/denoising_unet.pth -d /content/MusePose/pretrained_weights/MusePose -o denoising_unet.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/MusePose/motion_module.pth -d /content/MusePose/pretrained_weights/MusePose -o motion_module.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/MusePose/pose_guider.pth -d /content/MusePose/pretrained_weights/MusePose -o pose_guider.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/MusePose/reference_unet.pth -d /content/MusePose/pretrained_weights/MusePose -o reference_unet.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/dwpose/dw-ll_ucoco_384.pth -d /content/MusePose/pretrained_weights/dwpose -o dw-ll_ucoco_384.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/dwpose/yolox_l_8x8_300e_coco.pth -d /content/MusePose/pretrained_weights/dwpose -o yolox_l_8x8_300e_coco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/image_encoder/pytorch_model.bin -d /content/MusePose/pretrained_weights/image_encoder -o pytorch_model.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/raw/main/image_encoder/config.json -d /content/MusePose/pretrained_weights/image_encoder -o config.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/sd-image-variations-diffusers/diffusion_pytorch_model.bin -d /content/MusePose/pretrained_weights/sd-image-variations-diffusers/unet -o diffusion_pytorch_model.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/raw/main/sd-image-variations-diffusers/config.json -d /content/MusePose/pretrained_weights/sd-image-variations-diffusers/unet -o config.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/resolve/main/sd-vae-ft-mse/diffusion_pytorch_model.bin -d /content/MusePose/pretrained_weights/sd-vae-ft-mse -o diffusion_pytorch_model.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MusePose/raw/main/sd-vae-ft-mse/config.json -d /content/MusePose/pretrained_weights/sd-vae-ft-mse -o config.json

/content
fatal: destination path 'MusePose' already exists and is not an empty directory.
/content/MusePose
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122653 files and directories currently installed.)
Preparing to unpack .../00-libpython3.10-dev_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../01-libpython3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../02-python3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../03-libpython3.10-stdlib_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...


In [3]:
%cd /content/MusePose

!pip install -q -U av moviepy ninja openmim
!pip install -q -U setuptools
!mim install mmengine "mmcv>=2.0.1" "mmdet>=3.1.0" "mmpose>=1.1.0"
!pip install -q diffusers transformers accelerate einops omegaconf

!python -c "import av, mmcv, mmdet, mmpose; print('\n✅ All critical dependencies loaded successfully!')"

!python pose_align.py --imgfn_refer ./assets/images/ref.png --vidfn ./assets/videos/dance.mp4
!python test_stage_2.py --config ./configs/test_stage_2.yaml -W 448 -H 448

/content/MusePose
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 13.4.2 which is incompatible.
typer 0.27.1 requires rich>=13.8.0, but you have rich 13.4.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
openxlab 0.1.3 requires setuptools~=60.2.0, but you have setuptools 84.0.0 which is incompatible.
pytensor 2.38.3 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, bu

In [5]:
!python pose_align.py --imgfn_refer ./assets/images/ref.png --vidfn ./assets/videos/dance.mp4

height: 1920.0
width: 1080.0
fps: 24.0
/content/MusePose/pose/script/wholebody.py:9: UserWarning: The module 'mmcv' is not installed. The package will have limited functionality. Please install it using the command: mim install 'mmcv>=2.0.1'
  warnings.warn(
/content/MusePose/pose/script/wholebody.py:20: UserWarning: The module 'mmpose' is not installed. The package will have limited functionality. Please install it using the command: mim install 'mmpose>=1.1.0'
  warnings.warn(
/content/MusePose/pose/script/wholebody.py:27: UserWarning: The module 'mmdet' is not installed. The package will have limited functionality. Please install it using the command: mim install 'mmdet>=3.1.0'
  warnings.warn(
Traceback (most recent call last):
  File "/content/MusePose/pose_align.py", line 556, in <module>
    main()
    ~~~~^^
  File "/content/MusePose/pose_align.py", line 551, in main
    run_align_video_with_filterPose_translate_smooth(args)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^

In [3]:
!python test_stage_2.py --config ./configs/test_stage_2.yaml -W 448 -H 448

Traceback (most recent call last):
  File "/content/MusePose/test_stage_2.py", line 7, in <module>
    import av
ModuleNotFoundError: No module named 'av'


In [4]:
# 1. Install missing core media/video libraries
!pip install av moviepy packaging ninja

# 2. Install openmim to handle OpenMMLab packages
!pip install -U openmim

# 3. Install MMEngine, MMCV, MMDetection, and MMPose compatible with current PyTorch
!mim install mmengine
!mim install "mmcv>=2.0.1"
!mim install "mmdet>=3.1.0"
!mim install "mmpose>=1.1.0"

# 4. Install other required dependencies
!pip install diffusers transformers accelerate einops omegaconf

  Using cached av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.4/239.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.13/dist-packages/mim/__init__.py", line 10, in <module>
    import setuptools  # noqa: F401
    ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/setuptools/__init__.py", line 16, in <module>
    import setuptools.version
  File "/usr/local/lib/python3.13/dist-packages/setuptools/version.py", line 1, in <module>
    import pkg_resources
  File "/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py", line 2172, in <module>
    register_finder(pkgutil.ImpImporter, find_on_path)
                    ^^^^^^^^^^^^^^^^^^^
AttributeError: module 'pkgutil' has no attribute 'ImpImporter'. Did you mean: 'zipimporter'?
Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.13/dist-packages/mim/__init__.py", line 10, in <module>
    import setu

In [12]:
import os
import glob
import yaml
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

%cd /content/MusePose

# 1. Find the most recently uploaded image and video
img_files = [f for f in glob.glob("./assets/images/*.*") if f.endswith(('.png', '.jpg', '.jpeg'))]
vid_files = [f for f in glob.glob("./assets/videos/*.*") if f.endswith(('.mp4', '.avi', '.mov'))]

img_files.sort(key=os.path.getmtime, reverse=True)
vid_files.sort(key=os.path.getmtime, reverse=True)

my_img = img_files[0]
my_vid = vid_files[0]

print(f"✅ Selected Image: {my_img}")
print(f"✅ Selected Video: {my_vid}")
print("\n⏳ Running pose_align.py... (This may take a moment)")

# Run the alignment script
!python pose_align.py --imgfn_refer "{my_img}" --vidfn "{my_vid}"

# 2. Update test_stage_2.yaml programmatically
img_name = os.path.splitext(os.path.basename(my_img))[0]
vid_name = os.path.splitext(os.path.basename(my_vid))[0]
aligned_pose_path = f"./assets/poses/align/img_{img_name}_video_{vid_name}.mp4"

config_path = "./configs/test_stage_2.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['test_cases'] = {
    my_img: [aligned_pose_path]
}

with open(config_path, 'w') as f:
    yaml.dump(config, f)

print(f"\n✅ Updated YAML config to map:\n{my_img} -> {aligned_pose_path}")

# 3. Generate the Final Dance Video
print("\n⏳ Running test_stage_2.py... (This will take a while)")
!python test_stage_2.py --config ./configs/test_stage_2.yaml -W 448 -H 448

# 4. Display and Download the Result
output_videos = glob.glob("./output/**/*.mp4", recursive=True) + glob.glob("./outputs/**/*.mp4", recursive=True)
output_videos.sort(key=os.path.getmtime, reverse=True)

if output_videos:
    final_video = output_videos[0]
    print(f"🎉 Displaying generated video: {final_video}")
    files.download(final_video)
    mp4 = open(final_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="448" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("❌ No output video found. Please ensure test_stage_2.py finished without errors.")

/content/MusePose
✅ Selected Image: ./assets/images/Screenshot_20260828_142155.jpg
✅ Selected Video: ./assets/videos/Screenrecording_20260828_135147.mp4

⏳ Running pose_align.py... (This may take a moment)
height: 1612.0
width: 720.0
fps: 30.826864462537753
/content/MusePose/pose/script/wholebody.py:9: UserWarning: The module 'mmcv' is not installed. The package will have limited functionality. Please install it using the command: mim install 'mmcv>=2.0.1'
  warnings.warn(
/content/MusePose/pose/script/wholebody.py:20: UserWarning: The module 'mmpose' is not installed. The package will have limited functionality. Please install it using the command: mim install 'mmpose>=1.1.0'
  warnings.warn(
/content/MusePose/pose/script/wholebody.py:27: UserWarning: The module 'mmdet' is not installed. The package will have limited functionality. Please install it using the command: mim install 'mmdet>=3.1.0'
  warnings.warn(
Traceback (most recent call last):
  File "/content/MusePose/pose_align.py

In [7]:
!pip install diffusers==0.24.0

  Using cached diffusers-0.24.0-py3-none-any.whl.metadata (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.4 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.40.0
    Uninstalling diffusers-0.40.0:
      Successfully uninstalled diffusers-0.40.0


In [9]:
!pip install "huggingface_hub<0.26.0" "transformers<4.45.0" diffusers==0.24.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 14.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 120.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [10]:
!pip install --upgrade diffusers transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.24.0
    Uninstalling diffusers-0.24.0:
      Successfully uninstalled diffusers-0.24.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstal

In [11]:
import diffusers.models.embeddings as emb

if not hasattr(emb, "PositionNet"):
    if hasattr(emb, "GLIGENTextBoundingboxProjection"):
        emb.PositionNet = emb.GLIGENTextBoundingboxProjection
        print("✅ Successfully patched PositionNet alias in diffusers.")
    else:
        print("⚠️ GLIGENTextBoundingboxProjection not found.")

✅ Successfully patched PositionNet alias in diffusers.


In [13]:
# 1. Install mmengine directly via pip
!pip install mmengine

# 2. Install mmcv, mmdet, and mmpose using direct pre-built wheels
!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html || !pip install mmcv
!pip install mmdet mmpose

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 30.0 MB/s eta 0:00:00
Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.4/471.4 kB 13.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
/bin/bash: line 1: !pip: command not found
  Using cached mmdet-3.3.0-py3-none-any.whl.metadata (29 kB)
  U

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

# 1. Change to the MusePose directory
%cd /content/MusePose

# 2. Clean Uninstall existing problematic packages (OpenMMLab, diffusers, transformers, openmim, cocotools)
# Using '|| true' to prevent script from stopping if uninstall fails for a package not found
!pip uninstall -y mmcv mmdet mmpose mmengine openmim diffusers transformers accelerate setuptools xtcocotools pycocotools || true

/content/MusePose
Found existing installation: setuptools 60.2.0
Uninstalling setuptools-60.2.0:
  Successfully uninstalled setuptools-60.2.0
Found existing installation: pycocotools 2.0.11
Uninstalling pycocotools-2.0.11:
  Successfully uninstalled pycocotools-2.0.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.1/953.1 kB 18.3 MB/s  0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.


ERROR: Operation cancelled by user
^C
